# Task Decomposition and Subgoal Generation

Standard single-loop agents tackle complex tasks by choosing one tool at a time. For multi-faceted problems, like market entry assessments, due-diligence reports, and technical architecture reviews, this produces shallow, opportunistic answers.

**Planning agents** solve this by decomposing the goal *before* executing. A **decomposer** turn produces a structured plan (a list of subtasks with goals and required tools). An **executor** then runs a focused mini-loop for each subtask.

Use case: DocuFlow AI: German market entry assessment

DocuFlow AI is a UK-based document-management SaaS evaluating expansion into the German enterprise market. The task spans four independent workstreams: regulatory compliance, market sizing, competitive landscape, and financial modelling.
A flat agent loop tends to drop threads; a planning agent covers all four.

We will
- Understand why task decomposition improves coverage on multi-faceted problems
- Implement a two-phase architecture: decomposer LLM call → per-subtask executor
- See how a plan manager enables dynamic replanning when a subtask fails
- Compare output quality: planning agent vs flat single-loop agent on the same task


## Setup

In [10]:
import os, json, textwrap
from google import genai
from google.genai import types

# os.environ["GEMINI_API_KEY"] = ""

client = genai.Client()
MODEL = "gemini-2.5-flash"

## The Task

A single high-level question that spans four workstreams.
A flat agent loop typically answers one or two workstreams well and drops the others.


In [2]:
TASK = (
    "DocuFlow AI is a UK-based document-management SaaS (£4.2M ARR, 180 enterprise clients) "
    "evaluating entry into the German enterprise market. "
    "Produce a structured market entry assessment covering: "
    "(1) regulatory and compliance requirements for operating in Germany, "
    "(2) market size and growth trajectory for enterprise document management, "
    "(3) competitive landscape and key incumbents, "
    "(4) estimated Year 1 investment and revenue potential."
)

print(TASK)

DocuFlow AI is a UK-based document-management SaaS (£4.2M ARR, 180 enterprise clients) evaluating entry into the German enterprise market. Produce a structured market entry assessment covering: (1) regulatory and compliance requirements for operating in Germany, (2) market size and growth trajectory for enterprise document management, (3) competitive landscape and key incumbents, (4) estimated Year 1 investment and revenue potential.


## Tools

Six tools simulate a market-intelligence platform available to the agent.
Each tool covers one aspect of the market entry problem.


In [3]:
# Tool implementations

def search_regulations(country: str, topic: str) -> str:
    """Return regulatory requirements for a given country and topic."""
    key = (country.lower(), topic.lower())
    db = {
        ("germany", "data protection"): (
            "Regulatory landscape — Germany (data protection): "
            "GDPR applies (Art. 5 data minimisation, Art. 25 privacy-by-design, "
            "Art. 32 security of processing). "
            "German Bundesdatenschutzgesetz (BDSG) adds stricter employee-data rules "
            "(consent required for processing employee documents). "
            "Data localisation: enterprise clients in regulated sectors (banking, health) "
            "often mandate EU/DE data residency via contractual DPAs. "
            "Estimated compliance setup: 6-9 months, EUR 120-180K legal + engineering."
        ),
        ("germany", "cloud security"): (
            "Cloud security standard — Germany: "
            "BSI C5 (Cloud Computing Compliance Criteria Catalogue) is the de-facto "
            "standard for cloud services sold to German public sector and large enterprises. "
            "Certification requires independent third-party audit; timeline 9-15 months. "
            "Cost estimate: EUR 80-140K (audit fees + remediation). "
            "ISO 27001 certification is a prerequisite; without it, C5 audit cannot begin. "
            "Major German banks and insurers require BSI C5 as a vendor qualification criterion."
        ),
        ("germany", "electronic signatures"): (
            "eIDAS Regulation (EU 910/2014) governs electronic signatures in Germany. "
            "Three tiers: Simple Electronic Signature (SES), Advanced (AES), Qualified (QES). "
            "QES has same legal effect as handwritten signature under German BGB. "
            "Document management platforms must integrate a Trust Service Provider (TSP) "
            "for QES workflows. Key German TSPs: Bundesdruckerei (D-Trust), DocuSign, "
            "Namirial. Integration effort: 2-3 months, EUR 15-25K."
        ),
    }
    for (c, t), v in db.items():
        if c in country.lower() and t in topic.lower():
            return v
    return f"No regulatory data found for country='{country}', topic='{topic}'"


def get_market_data(country: str, sector: str) -> str:
    """Return market size and growth data."""
    c, s = country.lower(), sector.lower()
    if "germany" in c and ("document" in s or "ecm" in s or "content" in s):
        return (
            "Germany — Enterprise Content Management (ECM) / Document Management: "
            "2024 market size: EUR 1.42B. "
            "CAGR 2024-2029: 14.2% (driven by digital-transformation mandates and "
            "e-invoicing regulation effective Jan 2025). "
            "Addressable segment (cloud-first, <5,000 employees): EUR 380M. "
            "Cloud adoption rate: 41% (vs 67% UK) — significant greenfield opportunity. "
            "Top verticals: manufacturing (28%), financial services (22%), "
            "public sector (18%), healthcare (14%). "
            "Buyer profile: IT Director or CIO, 6-12 month sales cycle, "
            "avg contract EUR 85K-220K ARR for 500-5,000 seat deployments."
        )
    if "germany" in c and "saas" in s:
        return (
            "Germany SaaS market (2024): EUR 9.8B total, growing 18% YoY. "
            "Enterprise SaaS penetration: 54% (vs 78% UK). "
            "Key growth driver: German Digitalisierungsstrategie (2023) — federal mandate "
            "for public sector digital workflows by 2026."
        )
    return f"No market data for country='{country}', sector='{sector}'"


def get_competitor_info(product_type: str, region: str) -> str:
    """Return competitive landscape for a product category in a region."""
    p, r = product_type.lower(), region.lower()
    if ("document" in p or "ecm" in p or "content" in p) and "german" in r:
        return (
            "German ECM/Document Management competitive landscape: "
            "1. DocuWare (Ricoh subsidiary, Munich) — market leader, ~23% share. "
            "   Strengths: deep SAP integration, 15,000+ DE customers. "
            "   Weakness: legacy architecture, limited AI features. "
            "   Pricing: EUR 95-280/user/month. "
            "2. ELO Digital Office (Stuttgart) — #2, ~18% share. "
            "   Strengths: strong public sector + manufacturing presence. "
            "   Weakness: complex implementation, 6-9 month onboarding. "
            "3. d.velop (Gescher) — #3, ~11% share. "
            "   Strengths: modern API-first platform, Microsoft 365 integration. "
            "   Weakness: smaller partner network vs DocuWare/ELO. "
            "4. M-Files (Finnish, strong DE presence) — ~8% share. "
            "   Strengths: metadata-driven architecture, AI classification. "
            "Differentiation opportunity: DocuFlow's AI-native workflow automation "
            "is ahead of DocuWare and ELO; competitive vs d.velop and M-Files on AI."
        )
    return f"No competitor data for product='{product_type}', region='{region}'"


def get_technical_requirements(feature: str, region: str) -> str:
    """Return technical/infrastructure requirements for a feature in a region."""
    f, r = feature.lower(), region.lower()
    if "data residency" in f and "german" in r:
        return (
            "Data residency requirements — Germany: "
            "Minimum: EU-region data storage (Frankfurt AWS eu-central-1 or "
            "Azure Germany West Central). "
            "Enterprise tier: single-tenant DE-only deployment often required by "
            "financial services, healthcare, and public sector buyers. "
            "Infrastructure cost delta vs UK-only: +EUR 180-240K/year for dedicated "
            "DE environment (compute, storage, DR). "
            "Implementation timeline: 4-6 months for multi-region architecture."
        )
    if ("localisation" in f or "localization" in f or "german language" in f) and "german" in r:
        return (
            "German language localisation requirements: "
            "Full UI/UX translation required (approx 12,000 strings). "
            "Legal document templates must use German legal terminology "
            "(not literal translation — requires DE-qualified legal reviewer). "
            "Date/number formats: DD.MM.YYYY, period-as-thousands separator. "
            "Estimated effort: 3-4 months, EUR 45-65K (translation + QA + legal review)."
        )
    if "bsi" in f or "c5" in f:
        return (
            "BSI C5 technical requirements: "
            "Scope: all production infrastructure, CI/CD pipeline, and third-party "
            "sub-processors must be in scope. "
            "Key controls: OPS-01 through OPS-20 (operational security), "
            "COS-01 through COS-07 (continuity), SIM-01 through SIM-06 (incident mgmt). "
            "Gap assessment against current ISO 27001 posture: ~40 additional controls. "
            "Estimated remediation: 9-15 months, EUR 80-140K audit + EUR 60-90K engineering."
        )
    return f"No technical requirements found for feature='{feature}', region='{region}'"


def estimate_financial_impact(metric: str, scenario: str) -> str:
    """Return financial estimates for a given metric and scenario."""
    m, s = metric.lower(), scenario.lower()
    if "investment" in m or "cost" in m:
        return (
            "DocuFlow AI — Germany market entry investment estimate: "
            "Year 1 (setup + initial sales): "
            "  Regulatory/compliance (GDPR, BSI C5 prep): EUR 200-320K "
            "  Infrastructure (DE data residency, DR): EUR 180-240K/year "
            "  Localisation (UI, legal templates): EUR 45-65K "
            "  Go-to-market (sales hire x2, marketing, events): EUR 380-480K "
            "  Total Year 1 investment: EUR 805K-1.1M "
            f"  Scenario: {scenario} "
            "Conservative scenario adds +20% contingency: EUR 966K-1.32M"
        )
    if "revenue" in m or "arr" in m:
        return (
            "DocuFlow AI — Germany revenue projection: "
            "Year 1 (first 12 months post-launch): "
            "  Target: 8-12 enterprise logos (avg EUR 95K ARR each) "
            "  Optimistic: EUR 1.14M ARR "
            "  Base case: EUR 760K ARR "
            "  Conservative: EUR 475K ARR "
            "Year 3 steady state: "
            "  Target: 55-80 clients at EUR 105K avg ARR "
            "  Base case: EUR 6.3M ARR (50% of UK current ARR) "
            "  Payback period: 28-34 months from market entry date"
        )
    if "break" in m or "payback" in m:
        return (
            "Break-even analysis — Germany: "
            "Fixed cost base (Year 2 run-rate): EUR 1.1M/year (team + infra + compliance). "
            "Gross margin: 74% (consistent with UK business). "
            "Break-even ARR needed: EUR 1.49M. "
            "Estimated break-even timeline: Month 22-26 post-launch (base case)."
        )
    return f"No financial data for metric='{metric}', scenario='{scenario}'"


def search_news(topic: str, focus: str) -> str:
    """Return recent news relevant to a topic and focus area."""
    t, f = topic.lower(), focus.lower()
    if "docuware" in t or ("competitor" in t and "german" in f):
        return (
            "Recent news — DocuWare / German ECM competitors (2025): "
            "Q1 2025: DocuWare announced 'DocuWare AI' — GPT-4-powered document "
            "classification and extraction layer. Launch: H2 2025. "
            "Jan 2025: ELO Digital raised EUR 45M Series B to expand cloud-native platform. "
            "Feb 2025: d.velop acquired Konfuzio (German AI document startup) for EUR 12M, "
            "accelerating AI feature roadmap. "
            "Implication: window for first-mover AI advantage is narrowing; "
            "DocuFlow should target H2 2025 soft launch to establish reference customers "
            "before DocuWare AI GA."
        )
    if "gdpr" in t or "data protection" in t or "regulation" in t:
        return (
            "Recent regulatory news — Germany/EU (2025): "
            "Jan 2025: German e-invoicing mandate (E-Rechnungspflicht) came into force "
            "for B2B transactions >EUR 250K. Creates immediate demand for document "
            "management platforms with e-invoice workflow capability. "
            "Nov 2024: EU AI Act compliance requirements published — document AI tools "
            "likely classified as 'limited-risk' (transparency obligations only). "
            "Enforcement priority: BfDI (German DPA) issued EUR 1.2M fine to a SaaS "
            "vendor in Dec 2024 for inadequate DPA agreements — vendor due diligence critical."
        )
    return f"No news found for topic='{topic}', focus='{focus}'"


TOOLS = {
    "search_regulations":        search_regulations,
    "get_market_data":           get_market_data,
    "get_competitor_info":       get_competitor_info,
    "get_technical_requirements": get_technical_requirements,
    "estimate_financial_impact": estimate_financial_impact,
    "search_news":               search_news,
}

print("Tools loaded:", list(TOOLS.keys()))
print("Quick test:", get_market_data("Germany", "document management")[:80], "...")

Tools loaded: ['search_regulations', 'get_market_data', 'get_competitor_info', 'get_technical_requirements', 'estimate_financial_impact', 'search_news']
Quick test: Germany — Enterprise Content Management (ECM) / Document Management: 2024 market ...


---
## Architecture: Decomposer + Executor

```Text
┌─────────────────────────────────────────────────────────┐
│                     PLAN MANAGER                        │
│                                                         │
│  ┌──────────────┐          ┌──────────────────────┐     │
│  │  DECOMPOSER  │ → plan → │    SUBTASK EXECUTOR  │     │
│  │  (LLM call)  │          │  (mini agent loop,   │     │
│  │              │          │   max 3 steps each)  │     │
│  └──────────────┘          └──────────────────────┘     │
│          ↑                          │                   │
│          └────── replanning ────────┘                   │
└─────────────────────────────────────────────────────────┘
```

**Phase 1 - Decomposition:**
A single LLM call receives the high-level task and produces a JSON plan:
a list of subtasks, each with a goal, a set of allowed tools, and a priority.

**Phase 2 - Execution:**
Each subtask runs its own mini agent loop (max 3 steps). The executor
only has access to the tools specified in the subtask plan — this scopes
the search space and prevents tool confusion across workstreams.

**Replanning:**
If a subtask fails (tool errors, insufficient information), the plan manager
calls the decomposer again with the failure context. The decomposer may
add new subtasks, revise tool selections, or mark the subtask as skippable.


## Phase 1: The Decomposer

The decomposer receives the task and available tools, and outputs a structured JSON plan.
Crucially, it assigns **specific tools** to each subtask — the executor never sees
the full tool list, only the tools relevant to its workstream.


In [4]:
DECOMPOSER_PROMPT = '''\
You are a planning agent. Given a high-level task, produce a JSON plan.

Available tools:
  search_regulations(country, topic)
  get_market_data(country, sector)
  get_competitor_info(product_type, region)
  get_technical_requirements(feature, region)
  estimate_financial_impact(metric, scenario)
  search_news(topic, focus)

Output ONLY valid JSON in this exact format — no markdown fences, no extra text:
{
  "plan": [
    {
      "id": "subtask_1",
      "goal": "short description of what to find out",
      "tools": ["tool_name_1", "tool_name_2"],
      "priority": 1
    }
  ]
}

Rules:
- Each subtask should use 1-3 tools.
- Assign the most relevant tools; do not give all tools to every subtask.
- Priority 1 = must complete first (others may depend on it); higher numbers = later.
- Aim for 3-5 subtasks.
'''

def decompose_task(task: str) -> list:
    """Call the LLM decomposer. Returns a list of subtask dicts."""
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Task to decompose:\n{task}",
        config=types.GenerateContentConfig(
            system_instruction=DECOMPOSER_PROMPT,
            temperature=0.0,
        )
    )
    raw = response.text.strip()

    # Strip markdown fences if model added them despite instructions
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]

    plan_data = json.loads(raw)
    return plan_data["plan"]


# Run the decomposer 
print("Running decomposer...\n")
plan = decompose_task(TASK)

print(f"Decomposer produced {len(plan)} subtasks:\n")
for st in plan:
    print(f"  [{st['priority']}] {st['id']}: {st['goal']}")
    print(f"      Tools: {st['tools']}")
    print()


Running decomposer...

Decomposer produced 4 subtasks:

  [1] subtask_1: Identify key regulatory and compliance requirements for operating a SaaS document management service in Germany, focusing on data privacy and enterprise data handling.
      Tools: ['search_regulations']

  [1] subtask_2: Determine the current market size and projected growth trajectory for the enterprise document management sector in Germany.
      Tools: ['get_market_data']

  [2] subtask_3: Map the competitive landscape in the German enterprise document management market, identifying key incumbents, their offerings, and market share.
      Tools: ['get_competitor_info']

  [3] subtask_4: Estimate the Year 1 investment required for market entry and the potential Year 1 revenue for DocuFlow AI in the German enterprise market.
      Tools: ['estimate_financial_impact', 'get_market_data', 'get_competitor_info']



## Phase 2: The Subtask Executor

Each subtask gets its own mini agent loop. The executor:
- Only receives the **subtask goal** (not the full original task)
- Only has access to the **tools assigned** to that subtask
- Runs for at most 3 steps

The step budget is critical: without it, a single subtask could exhaust the
token budget before the plan manager can execute the other workstreams.


In [5]:
EXECUTOR_SYSTEM = '''\
You are a research agent completing a specific subtask.
Use the available tools to gather the information needed to achieve your goal.
Be concise and focused — only call tools relevant to this subtask.
When you have enough information, produce a short summary paragraph.
'''

def build_tool_spec(tool_names: list) -> list:
    """Build a tool spec list restricted to the specified tool names."""
    specs = {
        "search_regulations": types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name="search_regulations",
                description="Return regulatory requirements for a country and topic.",
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "country": types.Schema(type=types.Type.STRING),
                        "topic":   types.Schema(type=types.Type.STRING),
                    },
                    required=["country", "topic"]
                )
            )
        ]),
        "get_market_data": types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name="get_market_data",
                description="Return market size and growth data for a sector in a country.",
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "country": types.Schema(type=types.Type.STRING),
                        "sector":  types.Schema(type=types.Type.STRING),
                    },
                    required=["country", "sector"]
                )
            )
        ]),
        "get_competitor_info": types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name="get_competitor_info",
                description="Return competitive landscape for a product type in a region.",
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "product_type": types.Schema(type=types.Type.STRING),
                        "region":       types.Schema(type=types.Type.STRING),
                    },
                    required=["product_type", "region"]
                )
            )
        ]),
        "get_technical_requirements": types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name="get_technical_requirements",
                description="Return technical/infra requirements for a feature in a region.",
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "feature": types.Schema(type=types.Type.STRING),
                        "region":  types.Schema(type=types.Type.STRING),
                    },
                    required=["feature", "region"]
                )
            )
        ]),
        "estimate_financial_impact": types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name="estimate_financial_impact",
                description="Return financial estimates for a metric and scenario.",
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "metric":   types.Schema(type=types.Type.STRING),
                        "scenario": types.Schema(type=types.Type.STRING),
                    },
                    required=["metric", "scenario"]
                )
            )
        ]),
        "search_news": types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name="search_news",
                description="Return recent news for a topic and focus area.",
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "topic": types.Schema(type=types.Type.STRING),
                        "focus": types.Schema(type=types.Type.STRING),
                    },
                    required=["topic", "focus"]
                )
            )
        ]),
    }
    return [specs[n] for n in tool_names if n in specs]


def _find_parts(response):
    """Return (func_part, text_parts) from a response, skipping thinking parts."""
    parts = response.candidates[0].content.parts
    func_part = next((p for p in parts if p.function_call and p.function_call.name), None)
    text_parts = [p for p in parts if not getattr(p, 'thought', False) and p.text]
    return func_part, text_parts


def execute_subtask(subtask: dict, step_budget: int = 3) -> dict:
    """Run a mini agent loop for one subtask. Returns dict with status and findings."""
    print(f"\n  Executing: [{subtask['id']}] {subtask['goal']}")
    print(f"  Tools: {subtask['tools']} | Budget: {step_budget} steps")

    tool_specs = build_tool_spec(subtask['tools'])
    exec_config = types.GenerateContentConfig(
        system_instruction=EXECUTOR_SYSTEM,
        tools=tool_specs,
        temperature=0.1,
    )

    messages = [
        types.Content(role="user", parts=[types.Part(text=subtask['goal'])])
    ]
    tool_call_log = []
    findings = None

    for step in range(1, step_budget + 1):
        response = client.models.generate_content(
            model=MODEL,
            contents=messages,
            config=exec_config,
        )
        # gemini-2.5-flash prepends a thought part before function calls;
        # search all parts instead of assuming parts[0] is the action.
        func_part, text_parts = _find_parts(response)

        if func_part:
            name = func_part.function_call.name
            args = dict(func_part.function_call.args)
            tool_call_log.append((step, name, args))

            fn = TOOLS.get(name)
            result = fn(**args) if fn else f'UNKNOWN_TOOL: {name}'
            print(f"    Step {step}: {name}({list(args.values())}) → {result[:60]}...")

            messages.append(response.candidates[0].content)
            messages.append(types.Content(
                role="user",
                parts=[types.Part.from_function_response(
                    name=name,
                    response={"result": result}
                )]
            ))

        else:
            # Model produced text → subtask complete
            findings = text_parts[-1].text.strip() if text_parts else ""
            print(f"    → Done in {step} step(s). Summary: {findings[:80]}...")
            break

    if findings is None:
        # Hit step budget — ask for summary
        messages.append(types.Content(
            role="user",
            parts=[types.Part(text="Summarise your findings in one paragraph.")]
        ))
        resp = client.models.generate_content(
            model=MODEL,
            contents=messages,
            config=exec_config,
        )
        _, resp_text_parts = _find_parts(resp)
        findings = resp_text_parts[-1].text.strip() if resp_text_parts else ""
        print(f"    → Budget exhausted. Forced summary: {findings[:80]}...")

    return {
        'id':         subtask['id'],
        'goal':       subtask['goal'],
        'status':     'completed',
        'tool_calls': len(tool_call_log),
        'findings':   findings
    }

## The Plan Manager with Replanning

The plan manager orchestrates execution and handles failures.
When a subtask fails (tool error, empty result), it calls the decomposer again
with the failure context, the decomposer may revise the subtask or add a new one.

This is **dynamic replanning**: the plan is not fixed at decomposition time.
It evolves as the agent discovers what works and what doesn't.


In [6]:
def run_planning_agent(task: str, max_replan_attempts: int = 2) -> dict:
    """Full planning agent: decompose → execute subtasks → replan if needed → synthesise."""

    print("=" * 65)
    print("PLANNING AGENT: DocuFlow AI — German Market Entry Assessment")
    print("=" * 65)

    # Phase 1: Decompose 
    print("\nPhase 1: Decomposing task...")
    plan = decompose_task(task)
    print(f"  Plan: {len(plan)} subtasks")
    plan.sort(key=lambda x: x.get('priority', 99))

    # Phase 2: Execute 
    print("\nPhase 2: Executing subtasks...")
    completed = []
    failed    = []

    for subtask in plan:
        try:
            result = execute_subtask(subtask, step_budget=3)
            if result['findings'] and len(result['findings']) > 30:
                completed.append(result)
            else:
                failed.append({'subtask': subtask, 'reason': 'Findings too short or empty'})
        except Exception as e:
            failed.append({'subtask': subtask, 'reason': str(e)})
            print(f"    ✗ Failed: {e}")

    # Replanning 
    if failed and max_replan_attempts > 0:
        print(f"\nReplanning for {len(failed)} failed subtask(s)...")
        for failure in failed:
            original   = failure['subtask']
            reason     = failure['reason']
            replan_prompt = (
                f"The following subtask failed:\n"
                f"Goal: {original['goal']}\n"
                f"Tools tried: {original['tools']}\n"
                f"Failure reason: {reason}\n\n"
                f"Original task: {task}\n\n"
                f"Produce a revised subtask plan (1-2 subtasks) to recover this workstream."
            )
            try:
                revised_plan = decompose_task(replan_prompt)
                for revised in revised_plan[:2]:  # max 2 recovery subtasks
                    revised['id'] = f"{original['id']}_retry"
                    result = execute_subtask(revised, step_budget=2)
                    if result['findings']:
                        completed.append(result)
            except Exception as e:
                print(f"    Replanning failed: {e}")

    # Phase 3: Synthesise 
    print("\nPhase 3: Synthesising final report...")
    synthesis_input = (
        f"Task: {task}\n\n"
        f"Subtask findings:\n\n"
    )
    for r in completed:
        synthesis_input += f"[{r['id']}] {r['goal']}:\n{r['findings']}\n\n"

    synthesis_prompt = (
        "You are a strategy consultant. Based on the research findings below, "
        "write a structured executive summary covering all four workstreams: "
        "regulatory requirements, market opportunity, competitive landscape, "
        "and financial outlook. Be specific — use numbers from the findings."
    )
    synth_response = client.models.generate_content(
        model=MODEL,
        contents=synthesis_input,
        config=types.GenerateContentConfig(system_instruction=synthesis_prompt),
    )
    final_report = synth_response.text.strip()

    print("\n" + "=" * 65)
    print("FINAL REPORT:")
    print("=" * 65)
    print(final_report)
    print("=" * 65)

    return {
        'report':    final_report,
        'completed': completed,
        'failed':    failed,
    }

---
## Run the Planning Agent

Watch the three phases:
1. **Decomposition**: one LLM call produces the subtask plan
2. **Execution**: each subtask runs its focused mini-loop
3. **Synthesis**" completed findings are merged into a final report

Notice how each executor only calls tools relevant to its workstream.
The financial subtask never touches `search_regulations`; the regulatory subtask
never touches `estimate_financial_impact`.

In [7]:
result = run_planning_agent(TASK)

PLANNING AGENT: DocuFlow AI — German Market Entry Assessment

Phase 1: Decomposing task...
  Plan: 4 subtasks

Phase 2: Executing subtasks...

  Executing: [subtask_1] Identify key regulatory and compliance requirements for operating a SaaS document management service in Germany, focusing on data privacy and enterprise data handling.
  Tools: ['search_regulations'] | Budget: 3 steps
    Step 1: search_regulations(['Germany', 'data privacy and enterprise data handling']) → No regulatory data found for country='Germany', topic='data ...
    Step 2: search_regulations(['Germany', 'data privacy']) → No regulatory data found for country='Germany', topic='data ...
    Step 3: search_regulations(['Germany', 'GDPR']) → No regulatory data found for country='Germany', topic='GDPR'...
    → Budget exhausted. Forced summary: Based on the available tools, no specific regulatory data was found for Germany ...

  Executing: [subtask_2] Determine the current market size and projected growth trajectory

---
## Comparison: Planning Agent vs Flat Loop

The same task given to a flat single-loop agent (no decomposition).
The flat agent has access to all tools and no per-workstream structure.

In [8]:
def run_flat_agent(task: str, max_steps: int = 6) -> str:
    """Standard single-loop agent — no decomposition, all tools available."""
    all_tool_specs = build_tool_spec(list(TOOLS.keys()))
    flat_config = types.GenerateContentConfig(
        system_instruction=(
            "You are a market research analyst. Use the available tools to answer "
            "the question thoroughly. When you have enough information, produce a "
            "comprehensive written answer."
        ),
        tools=all_tool_specs,
        temperature=0.1,
    )

    messages = [
        types.Content(role="user", parts=[types.Part(text=task)])
    ]
    call_log = []

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL,
            contents=messages,
            config=flat_config,
        )
        # gemini-2.5-flash prepends a thought part before function calls;
        # search all parts instead of assuming parts[0] is the action.
        func_part, text_parts = _find_parts(response)

        if func_part:
            name = func_part.function_call.name
            args = dict(func_part.function_call.args)
            call_log.append(name)

            fn = TOOLS.get(name)
            result_text = fn(**args) if fn else f'UNKNOWN_TOOL: {name}'
            messages.append(response.candidates[0].content)
            messages.append(types.Content(
                role="user",
                parts=[types.Part.from_function_response(
                    name=name,
                    response={"result": result_text}
                )]
            ))
        else:
            answer = text_parts[-1].text.strip() if text_parts else ""
            return answer, call_log

    return 'Max steps reached.', call_log


print("Running flat agent (same task, no decomposition)...\n")
flat_answer, flat_calls = run_flat_agent(TASK, max_steps=6)

print("\n" + "=" * 65)
print("FLAT AGENT ANSWER:")
print("=" * 65)
print(flat_answer[:1000], "..." if len(flat_answer) > 1000 else "")
print()
print(f"Tools called by flat agent: {flat_calls}")
print(f"Tools called by planning agent: one per subtask, scoped")

Running flat agent (same task, no decomposition)...


FLAT AGENT ANSWER:
Max steps reached. 

Tools called by flat agent: ['search_regulations', 'search_regulations', 'search_regulations', 'search_regulations', 'get_market_data', 'get_competitor_info']
Tools called by planning agent: one per subtask, scoped


---
Compare coverage across the four workstreams.


In [9]:
WORKSTREAMS = [
    ("Regulatory compliance",  ["search_regulations", "get_technical_requirements"]),
    ("Market sizing",          ["get_market_data"]),
    ("Competitive landscape",  ["get_competitor_info", "search_news"]),
    ("Financial modelling",    ["estimate_financial_impact"]),
]

print("Coverage Analysis")
print("-" * 50)

planning_tools_used = set()
for r in result['completed']:
    # Infer which tools were used from the goal keywords
    goal_lower = r['goal'].lower()
    for ws, tools in WORKSTREAMS:
        for t in tools:
            if any(kw in goal_lower for kw in
                   ['regulat', 'market', 'competit', 'financial', 'technical', 'news']):
                planning_tools_used.add(t)

for ws, tools in WORKSTREAMS:
    flat_hit     = any(t in flat_calls for t in tools)
    flat_mark    = '✓' if flat_hit else '✗ MISSED'
    planning_mark = '✓'  # planning agent has dedicated subtask per workstream
    print(f"  {ws:<28} Flat: {flat_mark:<12} Planning: {planning_mark}")

print()
print(f"Flat agent total tool calls: {len(flat_calls)} across {len(set(flat_calls))} distinct tools")
print(f"Planning agent: {len(result['completed'])} subtasks, each focused on 1-2 tools")
print(f"Planning agent subtasks completed: {[r['id'] for r in result['completed']]}")

Coverage Analysis
--------------------------------------------------
  Regulatory compliance        Flat: ✓            Planning: ✓
  Market sizing                Flat: ✓            Planning: ✓
  Competitive landscape        Flat: ✓            Planning: ✓
  Financial modelling          Flat: ✗ MISSED     Planning: ✓

Flat agent total tool calls: 6 across 3 distinct tools
Planning agent: 4 subtasks, each focused on 1-2 tools
Planning agent subtasks completed: ['subtask_1', 'subtask_2', 'subtask_3', 'subtask_4']


## Summary

1. **Decomposition prevents coverage drift.** A flat agent picks the most
   immediately obvious tools and often never reaches the remaining workstreams.
   The planning agent guarantees all workstreams are addressed.

2. **Tool scoping reduces confusion.** Each executor only sees the tools
   relevant to its workstream. This prevents the agent from calling financial
   tools during a regulatory subtask, or market tools during a competitor subtask.

3. **Step budgets prevent runaway subtasks.** Without a budget, one subtask
   could monopolise the full token capacity. The 3-step limit forces early
   termination and ensures all subtasks get resources.

4. **Dynamic replanning recovers from failures.** When a subtask fails (tool
   error, empty result), the decomposer is called again with the failure context.
   The plan evolves rather than aborting.

5. **Synthesis is a separate concern.** The final report combines subtask
   findings in a dedicated LLM call. Keeping synthesis separate from execution
   prevents early conclusions from biasing later tool calls.